In [19]:
import pandas as pd 
from pathlib import Path
from bs4 import BeautifulSoup
import json
import html
 

In [2]:
test_file_path = Path("../fixtures/test2.html")

test_file_text = test_file_path.read_text(encoding="utf-8")



In [3]:
def load_ld_blocks(path):
    test_file_text = Path(path).read_text(encoding="utf-8")
    soup = BeautifulSoup(test_file_text, 'html.parser')

    ld_blocks = soup.find_all('script', type = "application/ld+json" )
    ld_parsed = []
    ld_flattened = []

    for block in ld_blocks:
        ld_flattened.append(block.get_text())
    for block in ld_flattened:
        parsed_block = json.loads(block)
        ld_parsed.append(parsed_block)
    return ld_parsed
blocks1 = load_ld_blocks("../fixtures/test1.html")
print(len(blocks1))

1


In [4]:
def find_recipe(graph):
    recipe = None
    for item in graph:
        if not isinstance(item, dict):
            continue
        item_type = item.get("@type", "none")
        if isinstance(item_type, str):
            match  = item_type == "Recipe"
        
        else:
            match = "Recipe" in item_type
        
        if match:
            recipe = item
    return recipe


In [5]:
def get_steps(recipe):

    all_steps = []
    for sections in recipe["recipeInstructions"]:
        if sections.get("@type") == "HowToStep":
            all_steps.append(sections['text'])    
        else:
            for step in sections['itemListElement']:
                all_steps.append(step['text'])
    return all_steps

In [6]:
def steps_from_page(path):
    blocks = load_ld_blocks(path)
    steps = get_steps(find_recipe(blocks[0]["@graph"]))
    return steps

In [7]:
steps1 = get_steps(find_recipe(blocks1[0]["@graph"]))
print(len(steps1))

14


In [8]:
print(len(steps_from_page("../fixtures/test1.html")))
print(len(steps_from_page("../fixtures/test2.html")))
print(len(steps_from_page("../fixtures/test3.html")))
print(len(steps_from_page("../fixtures/test4.html")))
print(len(steps_from_page("../fixtures/test5.html")))

14
8
9
11
5


In [52]:
recipe3 = find_recipe(load_ld_blocks("../fixtures/test1.html")[0]["@graph"])
recipe3["nutrition"]
print("nutrition" in recipe2)

False


In [54]:
recipe2 = find_recipe(load_ld_blocks("../fixtures/test5.html")[0]["@graph"])
recipe2["recipeInstructions"][0]

{'@type': 'HowToStep',
 'text': 'Preheat the oven to 325°F. Lightly spray an 8x8 baking dish (not a 9x9 dish or your brownies will overcook) with cooking spray and line it with parchment paper. Spray the parchment paper.',
 'name': 'Preheat the oven to 325°F. Lightly spray an 8x8 baking dish (not a 9x9 dish or your brownies will overcook) with cooking spray and line it with parchment paper. Spray the parchment paper.',
 'url': 'https://www.loveandlemons.com/brownies-recipe/#wprm-recipe-42107-step-0-0'}

In [11]:
blocks5 = load_ld_blocks("../fixtures/test5.html")
graph5 = blocks5[0]["@graph"]
print([type(item) for item in graph5])

[<class 'dict'>, <class 'dict'>, <class 'dict'>, <class 'dict'>, <class 'dict'>, <class 'dict'>, <class 'list'>, <class 'dict'>]


In [12]:
print(len(graph5[6]))

0


In [58]:
rows = []
for n in [1,2,3,4,5]:
    path = f"../fixtures/test{n}.html"
    blocks = load_ld_blocks(path)
    recipe = find_recipe(blocks[0]["@graph"])
    nutritionBool = "nutrition" in recipe
    rows.append({"Name": html.unescape(recipe["name"]),"Page ": n, "Step count": len(steps_from_page(path)), "Ingredients": len(recipe["recipeIngredient"]), "Has Nutrition": nutritionBool})

df = pd.DataFrame(rows)
df.to_csv("../results/extraction_results.csv", index=False)
df

,Name,Page,Step count,Ingredients,Has Nutrition
0,Macarons Recipe,1,14,13,True
1,Soft & Chewy Classic Snickerdoodle Cookies (On...,2,8,11,False
2,The Best Chocolate Chip Cookie Recipe Ever,3,9,10,True
3,Easy Chocolate Fudge Cake,4,11,13,True
4,Best Homemade Brownies,5,5,10,False


In [14]:
recipe1 = find_recipe(load_ld_blocks("../fixtures/test1.html")[0]["@graph"])
recipe1["name"]

'Macarons Recipe'

In [18]:
import html
html.unescape("Soft &amp; Chewy")

'Soft & Chewy'

In [39]:
recipe = find_recipe(blocks[0]["@graph"])
print(recipe["name"])

Macarons Recipe


In [59]:
check = pd.read_csv("../results/extraction_results.csv")
check

,Name,Page,Step count,Ingredients,Has Nutrition
0,Macarons Recipe,1,14,13,True
1,Soft & Chewy Classic Snickerdoodle Cookies (On...,2,8,11,False
2,The Best Chocolate Chip Cookie Recipe Ever,3,9,10,True
3,Easy Chocolate Fudge Cake,4,11,13,True
4,Best Homemade Brownies,5,5,10,False
